# Proyecto Integrador — Minería de Datos II

## Cloud Provider Analytics — Segunda Evaluación

**Institución:** ISTEA  
**Asignatura:** Minería de Datos II  
**Proyecto:** Cloud Provider Analytics  
**Integrante:** Nicolás Granata  
**Evaluación:** Segunda evaluación parcial — Implementación técnica y MVP end-to-end mínimo  
**Tecnología principal:** Apache Spark con PySpark  
**Entorno de trabajo:** Google Colab  

---

## Objetivo de esta entrega

La segunda evaluación busca transformar el diseño realizado en la primera instancia en una implementación técnica ejecutable y verificable.

El objetivo es construir un flujo mínimo que conecte:

**Landing → Bronze → Silver → Gold → Serving**

utilizando procesamiento batch y streaming, controles de calidad, Parquet y Cassandra/AstraDB.

A diferencia de la primera evaluación, las decisiones técnicas principales deberán quedar respaldadas por evidencia de ejecución y no únicamente por una propuesta conceptual.

## Alcance mínimo de la segunda evaluación

Durante este notebook se implementarán y documentarán los siguientes componentes:

| Componente | Objetivo |
|---|---|
| Correcciones E1 | Incorporar el feedback recibido y cerrar decisiones abiertas |
| Batch → Bronze | Ingestar al menos tres fuentes batch a Parquet con esquema explícito y metadata técnica |
| Streaming → Bronze | Procesar usage events con Structured Streaming |
| Silver | Limpiar, normalizar y enriquecer eventos y maestros |
| Calidad | Aplicar reglas verificables y separar registros inválidos en quarantine |
| Features | Calcular al menos tres métricas derivadas |
| Gold | Construir `org_daily_usage_by_service` |
| Serving | Publicar el mart en Cassandra/AstraDB |
| CQL | Ejecutar al menos dos consultas con resultados |
| Idempotencia | Demostrar re-ejecución sin duplicados |
| Gobierno | Registrar metadata, linaje, responsabilidades y controles |
| Reproducibilidad | Documentar ejecución, configuración, logs y pruebas |
| Backlog | Priorizar el trabajo restante para la instancia final |

# 1 - Incorporación del feedback de la primera evaluación

La primera evaluación obtuvo una calificación de **9/10** y validó la fundación general del proyecto.

El feedback recibido identifica principalmente decisiones que deberán pasar de un estado conceptual a uno comprobado durante esta segunda instancia.

| Feedback de E1 | Acción prevista en E2 |
|---|---|
| Algunos objetivos eran criterios de cobertura y no de éxito operacional | Relacionar la implementación con resultados y métricas verificables |
| La política de retención quedó abierta | Proponer una política concreta para el MVP y justificar sus límites |
| Watermark quedó como decisión abierta | Analizar los eventos y seleccionar el valor con evidencia |
| Particionamiento todavía conceptual | Evaluar alternativas y justificar la estrategia utilizada |
| Cassandra estaba planteado pero no implementado | Diseñar modelo query-first, cargar datos y ejecutar consultas reales |
| Estimación expresada solamente en porcentajes | Reemplazarla por horas-persona y responsable concreto |
| Idempotencia todavía conceptual | Re-ejecutar procesos y comparar conteos antes/después |
| Deduplicación principalmente propuesta | Implementarla y demostrar su resultado |

Las decisiones abiertas no se cerrarán mediante valores arbitrarios. Siempre que sea posible se utilizarán datos, conteos, resultados de ejecución o pruebas para justificar la decisión adoptada.

## 1.1 Criterio de evidencia

Durante esta evaluación se diferenciarán tres estados:

| Estado | Significado |
|---|---|
| **Medido** | Resultado obtenido directamente mediante exploración o ejecución |
| **Implementado** | Componente que existe y puede ejecutarse |
| **Decidido** | Elección técnica tomada a partir de evidencia documentada |

Cuando una decisión todavía no cuente con evidencia suficiente, se mantendrá explícitamente abierta.

Este criterio busca evitar presentar como comprobado aquello que todavía constituye únicamente una propuesta.

## 1.2 Criterios operacionales de éxito para E2

Para que esta segunda instancia pueda considerarse técnicamente cumplida se utilizarán criterios verificables:

- al menos tres fuentes batch deben existir en Bronze y conservar el conteo esperado luego de deduplicar por sus claves naturales;
- el streaming debe procesar los archivos JSONL mediante esquema explícito, watermark, deduplicación y checkpoint;
- Silver debe contener eventos conformados y enriquecidos con al menos un maestro;
- deben existir al menos tres features calculadas;
- las reglas de calidad deben producir conteos verificables y una quarantine inspeccionable;
- Gold debe respetar el grano `org_id + usage_date + service` sin claves duplicadas;
- Cassandra/AstraDB debe contener el mart y responder al menos dos consultas CQL;
- la re-ejecución no debe incrementar indebidamente los conteos;
- las decisiones de watermark, particionamiento y serving deben conservar evidencia de la razón utilizada.

# 2 - Preparación del entorno

La implementación se realizará en Google Colab utilizando Apache Spark en modo local.

Este entorno es suficiente para desarrollar y demostrar procesamiento batch, Parquet, Structured Streaming, watermarks, checkpoints, joins, agregaciones y controles de calidad.

La ejecución local permite validar la lógica utilizando las APIs de Spark trabajadas en clase, aunque no representa una infraestructura productiva.

In [2]:
!pip -q install pyspark==4.0.4

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2.1 Creación de la sesión de Spark

Se conserva la misma versión utilizada en la primera evaluación para mantener reproducibilidad entre ambas instancias.

`spark.sql.shuffle.partitions = 8` continúa siendo una configuración inicial adecuada para el tamaño reducido del dataset académico. Más adelante se evaluará el efecto del particionamiento sobre los archivos generados.

In [3]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
)

import os
import glob
import json
import math
import shutil
import zipfile
from pathlib import Path

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("cloud-provider-analytics-e2")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Spark: 4.0.4
Master: local[*]
Shuffle partitions: 8


## 2.2 Control de queries de streaming

Durante el desarrollo será habitual ejecutar nuevamente algunas celdas.

Una query de streaming que haya quedado activa puede interferir con una nueva ejecución. Se define una función auxiliar para detener las queries activas cuando sea necesario.

In [4]:
def detener_queries_activas():
    activas = spark.streams.active

    if not activas:
        print("No hay queries activas.")
        return

    for query in activas:
        print(f"Deteniendo: {query.name} | id={query.id}")
        query.stop()

    print("Todas las queries quedaron detenidas.")


detener_queries_activas()

No hay queries activas.


## 2.3 Carga del dataset

Se utilizará nuevamente el dataset original provisto por la cátedra:

`cloud_provider_challenge_dataset_v1.zip`

Los archivos de Landing representan las fuentes originales y continuarán tratándose como información inmutable.

Para evitar modificar estos archivos, las capas generadas durante E2 se almacenarán en rutas separadas dentro del entorno temporal de Google Colab.

In [5]:
from google.colab import files

uploaded = files.upload()

zip_candidates = [
    name for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

if len(zip_candidates) != 1:
    raise ValueError(
        "Se esperaba cargar un único archivo ZIP del dataset."
    )

zip_name = zip_candidates[0]
print("Archivo cargado:", zip_name)

Saving cloud_provider_challenge_dataset_v1.zip to cloud_provider_challenge_dataset_v1.zip
Archivo cargado: cloud_provider_challenge_dataset_v1.zip


In [6]:
DATASET_ROOT = "/content/cloud_provider_dataset"

# Se limpia únicamente la copia temporal del dataset.
if os.path.exists(DATASET_ROOT):
    shutil.rmtree(DATASET_ROOT)

os.makedirs(DATASET_ROOT, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(DATASET_ROOT)

LANDING_PATH = f"{DATASET_ROOT}/datalake/landing"

if not os.path.isdir(LANDING_PATH):
    raise FileNotFoundError(
        f"No se encontró Landing en {LANDING_PATH}"
    )

print("Dataset extraído en:", DATASET_ROOT)
print("Landing:", LANDING_PATH)

Dataset extraído en: /content/cloud_provider_dataset
Landing: /content/cloud_provider_dataset/datalake/landing


## 2.4 Rutas de trabajo de E2

Se separará físicamente la fuente original de las capas producidas por el pipeline.

La estructura temporal utilizada será:

`Landing → Bronze → Silver → Gold`

Además se utilizarán rutas separadas para quarantine, checkpoints, pruebas y evidencias.

Esta separación permite mantener Landing inmutable y facilita las pruebas de re-ejecución e idempotencia.

In [7]:
PROJECT_ROOT = "/content/cloud_provider_e2"

BRONZE_PATH = f"{PROJECT_ROOT}/datalake/bronze"
SILVER_PATH = f"{PROJECT_ROOT}/datalake/silver"
GOLD_PATH = f"{PROJECT_ROOT}/datalake/gold"
QUARANTINE_PATH = f"{PROJECT_ROOT}/datalake/quarantine"
CHECKPOINT_PATH = f"{PROJECT_ROOT}/checkpoints"
EVIDENCE_PATH = f"{PROJECT_ROOT}/evidence"
TEST_PATH = f"{PROJECT_ROOT}/tests"
CQL_PATH = f"{PROJECT_ROOT}/cql"

paths = [
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    QUARANTINE_PATH,
    CHECKPOINT_PATH,
    EVIDENCE_PATH,
    TEST_PATH,
    CQL_PATH,
]

for path in paths:
    os.makedirs(path, exist_ok=True)

print("Rutas de E2 preparadas:")
for path in paths:
    print("-", path)

Rutas de E2 preparadas:
- /content/cloud_provider_e2/datalake/bronze
- /content/cloud_provider_e2/datalake/silver
- /content/cloud_provider_e2/datalake/gold
- /content/cloud_provider_e2/datalake/quarantine
- /content/cloud_provider_e2/checkpoints
- /content/cloud_provider_e2/evidence
- /content/cloud_provider_e2/tests
- /content/cloud_provider_e2/cql


In [8]:
csv_files = sorted(glob.glob(f"{LANDING_PATH}/*.csv"))
event_files = sorted(
    glob.glob(f"{LANDING_PATH}/usage_events_stream/*.jsonl")
)

print("Archivos CSV encontrados:", len(csv_files))
for file_path in csv_files:
    print(" -", os.path.basename(file_path))

print("\nArchivos JSONL de eventos:", len(event_files))
print("Primer JSONL:", os.path.basename(event_files[0]) if event_files else None)
print("Último JSONL:", os.path.basename(event_files[-1]) if event_files else None)

assert len(csv_files) == 7, "Se esperaban 7 fuentes CSV."
assert len(event_files) == 120, "Se esperaban 120 archivos JSONL."

Archivos CSV encontrados: 7
 - billing_monthly.csv
 - customers_orgs.csv
 - marketing_touches.csv
 - nps_surveys.csv
 - resources.csv
 - support_tickets.csv
 - users.csv

Archivos JSONL de eventos: 120
Primer JSONL: events_part_0000.jsonl
Último JSONL: events_part_0119.jsonl


## 2.5 Estado inicial de la ejecución

La validación inicial confirma que:

- las siete fuentes CSV están disponibles;
- existen 120 archivos JSONL de eventos;
- Landing permanece separada de las capas generadas;
- Bronze, Silver, Gold y quarantine poseen rutas independientes;
- los checkpoints no se comparten con las capas de datos.

A partir de este punto comienza la implementación del pipeline.

# 3 - Ingesta batch hacia Bronze

La primera implementación productiva corresponde a tres fuentes batch solicitadas como referencia por la consigna:

- `customers_orgs.csv`;
- `users.csv`;
- `billing_monthly.csv`.

En todos los casos se evitará `inferSchema`. La lectura utilizará esquemas explícitos y posteriormente se aplicarán conversiones controladas de tipos.

Bronze conservará el grano de cada fuente, agregará metadata técnica y realizará únicamente deduplicación por clave natural cuando corresponda.

## 3.1 Esquemas explícitos de entrada

Para los CSV se utiliza un esquema de entrada explícito que conserva inicialmente las representaciones textuales de la fuente.

La tipificación definitiva se realiza de forma controlada en la misma transformación hacia Bronze. Esto evita depender de inferencias automáticas de Spark y hace visibles las reglas aplicadas a fechas, booleanos y valores numéricos.

In [9]:
customers_raw_schema = StructType([
    StructField("org_id", StringType(), True),
    StructField("org_name", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("hq_region", StringType(), True),
    StructField("plan_tier", StringType(), True),
    StructField("is_enterprise", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("sales_rep", StringType(), True),
    StructField("lifecycle_stage", StringType(), True),
    StructField("marketing_source", StringType(), True),
    StructField("nps_score", StringType(), True),
])

users_raw_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("role", StringType(), True),
    StructField("active", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("last_login", StringType(), True),
])

billing_raw_schema = StructType([
    StructField("invoice_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("month", StringType(), True),
    StructField("subtotal", StringType(), True),
    StructField("credits", StringType(), True),
    StructField("taxes", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("exchange_rate_to_usd", StringType(), True),
])

## 3.2 Funciones comunes de metadata y normalización

A partir de Bronze se incorporan dos columnas técnicas definidas en E1:

- `source_file`: archivo de origen;
- `ingest_ts`: momento de incorporación al pipeline.

También se deriva `ingest_date` para las fuentes maestras pequeñas. Esta partición tiene baja cardinalidad y permite distinguir ejecuciones sin utilizar identificadores de negocio como particiones.

In [10]:
def add_technical_columns(df):
    return (
        df
        .withColumn("source_file", F.input_file_name())
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("ingest_date", F.current_date())
    )


def parse_bool(column_name):
    normalized = F.lower(F.trim(F.col(column_name)))
    return (
        F.when(normalized.isin("true", "1", "yes", "si", "sí"), F.lit(True))
        .when(normalized.isin("false", "0", "no"), F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))
    )

## 3.3 Customers a Bronze

`customers_orgs` conserva una fila por organización y utiliza `org_id` como clave natural.

Se tipifican `is_enterprise`, `signup_date` y `nps_score`, se agregan columnas técnicas y se deduplica por `org_id`.

In [11]:
customers_raw = (
    spark.read
    .option("header", True)
    .schema(customers_raw_schema)
    .csv(f"{LANDING_PATH}/customers_orgs.csv")
)

customers_bronze = (
    add_technical_columns(customers_raw)
    .withColumn("is_enterprise", parse_bool("is_enterprise"))
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn(
        "nps_score",
        F.when(
            F.trim(F.col("nps_score")) == "",
            F.lit(None).cast("int")
        ).otherwise(
            F.col("nps_score").cast("double").cast("int")
        )
    )
    .dropDuplicates(["org_id"])
)

customers_before = customers_raw.count()
customers_after = customers_bronze.count()

print("Customers Landing:", customers_before)
print("Customers Bronze:", customers_after)
print("Duplicados eliminados:", customers_before - customers_after)
customers_bronze.printSchema()

Customers Landing: 80
Customers Bronze: 80
Duplicados eliminados: 0
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: integer (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingest_ts: timestamp (nullable = false)
 |-- ingest_date: date (nullable = false)



In [12]:
CUSTOMERS_BRONZE_PATH = f"{BRONZE_PATH}/customers_orgs"

(
    customers_bronze
    .write
    .mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(CUSTOMERS_BRONZE_PATH)
)

customers_bronze_check = spark.read.parquet(CUSTOMERS_BRONZE_PATH)
print("Registros re-leídos desde Bronze:", customers_bronze_check.count())
customers_bronze_check.show(5, truncate=False)

Registros re-leídos desde Bronze: 80
+------------+--------------+----------+----------+----------+-------------+-----------+---------+---------------+----------------+---------+--------------------------------------------------------------------------+--------------------------+-----------+
|org_id      |org_name      |industry  |hq_region |plan_tier |is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|source_file                                                               |ingest_ts                 |ingest_date|
+------------+--------------+----------+----------+----------+-------------+-----------+---------+---------------+----------------+---------+--------------------------------------------------------------------------+--------------------------+-----------+
|org_0lvsnujz|Nova Cloud 45 |Retail    |us-east   |standard  |false        |2025-05-19 |rep_e    |prospect       |event           |-1       |file:///content/cloud_provider_dataset/datalake/landin

## 3.4 Users a Bronze

`users` conserva una fila por usuario y utiliza `user_id` como clave natural.

Se tipifican `active`, `created_at` y `last_login`, manteniendo la referencia a `org_id` para posteriores joins.

In [13]:
users_raw = (
    spark.read
    .option("header", True)
    .schema(users_raw_schema)
    .csv(f"{LANDING_PATH}/users.csv")
)

users_bronze = (
    add_technical_columns(users_raw)
    .withColumn("active", parse_bool("active"))
    .withColumn(
        "created_at",
        F.to_date(
            F.when(
                F.trim(F.col("created_at")) == "",
                F.lit(None)
            ).otherwise(F.col("created_at"))
        )
    )
    .withColumn(
        "last_login",
        F.to_date(
            F.when(
                F.trim(F.col("last_login")) == "",
                F.lit(None)
            ).otherwise(F.col("last_login"))
        )
    )
    .dropDuplicates(["user_id"])
)

users_before = users_raw.count()
users_after = users_bronze.count()

print("Users Landing:", users_before)
print("Users Bronze:", users_after)
print("Duplicados eliminados:", users_before - users_after)
users_bronze.printSchema()

Users Landing: 800
Users Bronze: 800
Duplicados eliminados: 0
root
 |-- user_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- role: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_login: date (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingest_ts: timestamp (nullable = false)
 |-- ingest_date: date (nullable = false)



In [14]:
USERS_BRONZE_PATH = f"{BRONZE_PATH}/users"

(
    users_bronze
    .write
    .mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(USERS_BRONZE_PATH)
)

users_bronze_check = spark.read.parquet(USERS_BRONZE_PATH)
print("Registros re-leídos desde Bronze:", users_bronze_check.count())
users_bronze_check.show(5, truncate=False)

Registros re-leídos desde Bronze: 800
+-------------+------------+-------------------------+-------------+------+----------+----------+-----------------------------------------------------------------+--------------------------+-----------+
|user_id      |org_id      |email                    |role         |active|created_at|last_login|source_file                                                      |ingest_ts                 |ingest_date|
+-------------+------------+-------------------------+-------------+------+----------+----------+-----------------------------------------------------------------+--------------------------+-----------+
|user_01nouhtc|org_1swjckjl|user_01nouhtc@example.com|developer    |true  |2025-07-12|2025-06-05|file:///content/cloud_provider_dataset/datalake/landing/users.csv|2026-11-18 18:17:54.759833|2026-11-18 |
|user_0224wk0y|org_trp0j43d|user_0224wk0y@example.com|admin        |true  |2025-06-17|2025-07-29|file:///content/cloud_provider_dataset/datalake/landi

## 3.5 Billing mensual a Bronze

`billing_monthly` utiliza `invoice_id` como clave natural.

Los importes y el tipo de cambio se convierten explícitamente a valores numéricos. `month` se conserva como período `YYYY-MM` y también se deriva `month_date` como fecha del primer día del mes.

En este caso la partición por `month` es coherente con la naturaleza mensual de la fuente y produce una cardinalidad reducida.

In [15]:
billing_raw = (
    spark.read
    .option("header", True)
    .schema(billing_raw_schema)
    .csv(f"{LANDING_PATH}/billing_monthly.csv")
)

billing_bronze = (
    add_technical_columns(billing_raw)
    .withColumn(
        "month",
        F.to_date(F.col("month"))
    )
    .withColumn(
        "subtotal",
        F.when(
            F.trim(F.col("subtotal")) == "",
            F.lit(None).cast("double")
        ).otherwise(
            F.col("subtotal").cast("double")
        )
    )
    .withColumn(
        "credits",
        F.when(
            F.trim(F.col("credits")) == "",
            F.lit(None).cast("double")
        ).otherwise(
            F.col("credits").cast("double")
        )
    )
    .withColumn(
        "taxes",
        F.when(
            F.trim(F.col("taxes")) == "",
            F.lit(None).cast("double")
        ).otherwise(
            F.col("taxes").cast("double")
        )
    )
    .withColumn(
        "exchange_rate_to_usd",
        F.when(
            F.trim(F.col("exchange_rate_to_usd")) == "",
            F.lit(None).cast("double")
        ).otherwise(
            F.col("exchange_rate_to_usd").cast("double")
        )
    )
    .dropDuplicates(["invoice_id"])
)

billing_before = billing_raw.count()
billing_after = billing_bronze.count()

print("Billing Landing:", billing_before)
print("Billing Bronze:", billing_after)
print("Duplicados eliminados:", billing_before - billing_after)
billing_bronze.printSchema()

Billing Landing: 240
Billing Bronze: 240
Duplicados eliminados: 0
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- credits: double (nullable = true)
 |-- taxes: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: double (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingest_ts: timestamp (nullable = false)
 |-- ingest_date: date (nullable = false)



In [16]:
BILLING_BRONZE_PATH = f"{BRONZE_PATH}/billing_monthly"

(
    billing_bronze
    .write
    .mode("overwrite")
    .partitionBy("month")
    .parquet(BILLING_BRONZE_PATH)
)

billing_bronze_check = spark.read.parquet(BILLING_BRONZE_PATH)
print("Registros re-leídos desde Bronze:", billing_bronze_check.count())
billing_bronze_check.show(5, truncate=False)

Registros re-leídos desde Bronze: 240
+------------+------------+--------+-------+------+--------+--------------------+---------------------------------------------------------------------------+-------------------------+-----------+----------+
|invoice_id  |org_id      |subtotal|credits|taxes |currency|exchange_rate_to_usd|source_file                                                                |ingest_ts                |ingest_date|month     |
+------------+------------+--------+-------+------+--------+--------------------+---------------------------------------------------------------------------+-------------------------+-----------+----------+
|inv_03ivwack|org_hv3a3zmf|350.32  |NULL   |73.57 |ARS     |0.00158             |file:///content/cloud_provider_dataset/datalake/landing/billing_monthly.csv|2026-11-18 18:17:57.55729|2026-11-18 |2025-06-01|
|inv_07uj4fqk|org_1n13jcat|1079.91 |2.3    |226.78|ARS     |0.00153             |file:///content/cloud_provider_dataset/datalake/landi

## 3.6 Evidencia consolidada de Batch → Bronze

La evidencia se construye leyendo nuevamente los archivos Parquet generados. De este modo no se valida solamente el DataFrame en memoria, sino la salida física de Bronze.

In [17]:
batch_bronze_summary = spark.createDataFrame([
    (
        "customers_orgs",
        customers_before,
        customers_bronze_check.count(),
        customers_before - customers_bronze_check.count(),
        "org_id",
        "ingest_date",
    ),
    (
        "users",
        users_before,
        users_bronze_check.count(),
        users_before - users_bronze_check.count(),
        "user_id",
        "ingest_date",
    ),
    (
        "billing_monthly",
        billing_before,
        billing_bronze_check.count(),
        billing_before - billing_bronze_check.count(),
        "invoice_id",
        "month",
    ),
], [
    "fuente",
    "landing_rows",
    "bronze_rows",
    "duplicados_eliminados",
    "clave_natural",
    "particion",
])

batch_bronze_summary.show(truncate=False)

+---------------+------------+-----------+---------------------+-------------+-----------+
|fuente         |landing_rows|bronze_rows|duplicados_eliminados|clave_natural|particion  |
+---------------+------------+-----------+---------------------+-------------+-----------+
|customers_orgs |80          |80         |0                    |org_id       |ingest_date|
|users          |800         |800        |0                    |user_id      |ingest_date|
|billing_monthly|240         |240        |0                    |invoice_id   |month      |
+---------------+------------+-----------+---------------------+-------------+-----------+



### Resultado esperado de este bloque

Con el dataset de la cátedra se espera conservar 80 organizaciones, 800 usuarios y 240 facturas, debido a que en E1 no se detectaron duplicados en las claves naturales analizadas.

Si los conteos difieren, la diferencia deberá investigarse antes de continuar hacia Silver.

# 4 - Análisis del tiempo de evento y decisión del watermark

En E1 el watermark quedó deliberadamente abierto. El feedback recibido solicita convertir esa propuesta en una decisión basada en evidencia.

El dataset no posee un timestamp productivo de llegada. Por ese motivo se utilizará como aproximación el orden de los archivos JSONL entregados, que representan micro-lotes simulados.

La medición no pretende afirmar una latencia real de producción. Su objetivo es obtener una tolerancia razonable para el MVP a partir de la evidencia disponible.

## 4.1 Esquema explícito de usage events

El esquema refleja las trece columnas observadas durante E1.

`value` se mantiene inicialmente como texto porque el dataset contiene representaciones heterogéneas. La conversión numérica controlada se realizará en Silver.

In [18]:
usage_event_schema = StructType([
    StructField("carbon_kg", DoubleType(), True),
    StructField("cost_usd_increment", DoubleType(), True),
    StructField("event_id", StringType(), True),
    StructField("genai_tokens", LongType(), True),
    StructField("metric", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("schema_version", LongType(), True),
    StructField("service", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("value", StringType(), True),
])

## 4.2 Lectura estática auxiliar para medir lateness

La medición del watermark se realiza antes de iniciar el stream.

Se calcula para cada evento cuánto se encuentra por detrás del máximo `event_time` observado en micro-lotes anteriores según el orden de archivo.

Esta es una aproximación académica a la noción de llegada tardía. La limitación queda documentada porque el dataset no contiene un `arrival_timestamp` real.

In [19]:
events_for_watermark = (
    spark.read
    .schema(usage_event_schema)
    .json(f"{LANDING_PATH}/usage_events_stream/*.jsonl")
    .withColumn("source_file", F.input_file_name())
    .withColumn("file_name", F.regexp_extract("source_file", r"([^/]+)$", 1))
    .withColumn("event_time", F.to_timestamp("timestamp"))
)

# El nombre de archivo se utiliza como orden de llegada simulado.
file_stats = (
    events_for_watermark
    .groupBy("file_name")
    .agg(
        F.min("event_time").alias("file_min_event_time"),
        F.max("event_time").alias("file_max_event_time"),
        F.count("*").alias("file_rows"),
    )
)

file_order_window = (
    Window
    .orderBy("file_name")
    .rowsBetween(Window.unboundedPreceding, -1)
)

file_stats = (
    file_stats
    .withColumn(
        "max_event_time_seen_before",
        F.max("file_max_event_time").over(file_order_window)
    )
)

lateness = (
    events_for_watermark
    .join(
        file_stats.select(
            "file_name",
            "max_event_time_seen_before"
        ),
        "file_name",
        "left"
    )
    .withColumn(
        "lateness_seconds",
        F.when(
            F.col("max_event_time_seen_before").isNull(),
            F.lit(0.0)
        ).otherwise(
            F.greatest(
                F.lit(0.0),
                F.col("max_event_time_seen_before").cast("long")
                - F.col("event_time").cast("long")
            )
        )
    )
)

print("Eventos analizados:", lateness.count())
file_stats.orderBy("file_name").show(10, truncate=False)

Eventos analizados: 43200
+----------------------+-------------------+-------------------+---------+--------------------------+
|file_name             |file_min_event_time|file_max_event_time|file_rows|max_event_time_seen_before|
+----------------------+-------------------+-------------------+---------+--------------------------+
|events_part_0000.jsonl|2025-07-03 00:08:00|2025-08-31 16:32:00|360      |NULL                      |
|events_part_0001.jsonl|2025-07-03 02:38:00|2025-08-31 23:51:00|360      |2025-08-31 16:32:00       |
|events_part_0002.jsonl|2025-07-03 14:16:00|2025-08-31 20:47:00|360      |2025-08-31 23:51:00       |
|events_part_0003.jsonl|2025-07-03 00:48:00|2025-08-31 23:51:00|360      |2025-08-31 23:51:00       |
|events_part_0004.jsonl|2025-07-03 10:43:00|2025-08-31 20:40:00|360      |2025-08-31 23:51:00       |
|events_part_0005.jsonl|2025-07-03 01:42:00|2025-08-31 23:56:00|360      |2025-08-31 23:51:00       |
|events_part_0006.jsonl|2025-07-03 07:13:00|2025-08-31 2

## 4.3 Comparación de tolerancias candidatas

Se comparan varios valores de watermark y se mide qué porcentaje de eventos quedaría dentro de la tolerancia observada.

Para el MVP se elegirá el menor valor candidato que cubra al menos el 99 % de los eventos según esta aproximación.

El criterio busca evitar tanto una tolerancia demasiado pequeña como un estado innecesariamente prolongado.

In [62]:
total_events_for_watermark = lateness.count()
watermark_candidates_minutes = [
    120,      # 2 horas
    1440,     # 1 día
    10080,    # 7 días
    43200,    # 30 días
    86400,    # 60 días
]
watermark_results = []

for minutes in watermark_candidates_minutes:
    too_late = (
        lateness
        .filter(F.col("lateness_seconds") > minutes * 60)
        .count()
    )

    retained = total_events_for_watermark - too_late
    retained_pct = (
        100.0 * retained / total_events_for_watermark
        if total_events_for_watermark
        else 0.0
    )

    watermark_results.append(
        (minutes, too_late, retained, retained_pct)
    )

watermark_evidence = spark.createDataFrame(
    watermark_results,
    [
        "watermark_minutes",
        "eventos_fuera_tolerancia",
        "eventos_dentro_tolerancia",
        "porcentaje_cubierto",
    ]
)

watermark_evidence.orderBy("watermark_minutes").show(truncate=False)

eligible = [
    row[0]
    for row in watermark_results
    if row[3] >= 99.0
]

WATERMARK_MINUTES = (
    min(eligible)
    if eligible
    else max(watermark_candidates_minutes)
)

print(
    "Watermark seleccionado para el MVP:",
    WATERMARK_MINUTES,
    "minutes"
)

+-----------------+------------------------+-------------------------+-------------------+
|watermark_minutes|eventos_fuera_tolerancia|eventos_dentro_tolerancia|porcentaje_cubierto|
+-----------------+------------------------+-------------------------+-------------------+
|120              |42783                   |417                      |0.9652777777777778 |
|1440             |42118                   |1082                     |2.5046296296296298 |
|10080            |37831                   |5369                     |12.42824074074074  |
|43200            |21421                   |21779                    |50.414351851851855 |
|86400            |0                       |43200                    |100.0              |
+-----------------+------------------------+-------------------------+-------------------+

Watermark seleccionado para el MVP: 86400 minutes


In [63]:
quantiles = lateness.approxQuantile(
    "lateness_seconds",
    [0.50, 0.95, 0.99, 1.00],
    0.01
)

print("Lateness aproximada:")
print("P50:", round(quantiles[0] / 60, 2), "min")
print("P95:", round(quantiles[1] / 60, 2), "min")
print("P99:", round(quantiles[2] / 60, 2), "min")
print("Máximo aproximado:", round(quantiles[3] / 60, 2), "min")

Lateness aproximada:
P50: 42608.0 min
P95: 81269.0 min
P99: 86394.0 min
Máximo aproximado: 86394.0 min


## 4.4 Decisión del watermark

**Estado:** decidido a partir de evidencia del dataset académico.

Se evaluaron tolerancias de 120 minutos, 1 día, 7 días, 30 días y 60 días.

Los resultados mostraron que una tolerancia de 30 días cubriría aproximadamente el 50,41 % de los eventos, mientras que 60 días cubre el 100 % de los 43.200 eventos observados.

La lateness aproximada obtenida fue:

- P50: 42.608 minutos;
- P95: 81.269 minutos;
- P99: 86.394 minutos;
- máximo observado: 86.394 minutos.

Por este motivo se seleccionó un watermark de **86.400 minutos (60 días)** para el MVP académico.

La ejecución posterior de Structured Streaming procesó los **43.200 eventos**, conservó **43.200 `event_id` distintos** y no descartó registros por watermark.

### Limitación

El dataset no dispone de un timestamp real de llegada a una infraestructura productiva y los archivos entregados no representan un orden cronológico estricto de los eventos.

Por lo tanto, el valor de 60 días no se propone como configuración productiva. Se utiliza en este MVP para evitar descartar artificialmente eventos válidos del dataset académico y debería recalibrarse en un entorno real utilizando métricas de lateness observadas en producción.

# 5 - Ingesta streaming hacia Bronze

`usage_events_stream/*.jsonl` será procesado con Structured Streaming utilizando el patrón de file streaming trabajado en clase.

El stream incorpora:

- esquema explícito;
- event time;
- watermark;
- deduplicación por `event_id`;
- metadata técnica;
- checkpoint independiente;
- salida Parquet particionada.

Se utilizará `availableNow=True` para procesar todos los micro-lotes disponibles y detener la query al finalizar. Esta modalidad permite una demostración reproducible sobre un dataset estático que simula llegada incremental.

## 5.1 Preparación del stream

`maxFilesPerTrigger = 10` limita la cantidad de archivos incorporados en cada micro-lote y permite observar procesamiento incremental aun cuando los 120 archivos ya estén disponibles.

In [64]:
EVENTS_BRONZE_PATH = f"{BRONZE_PATH}/usage_events"
EVENTS_CHECKPOINT = f"{CHECKPOINT_PATH}/bronze_usage_events"

usage_stream = (
    spark.readStream
    .schema(usage_event_schema)
    .option("maxFilesPerTrigger", 10)
    .json(f"{LANDING_PATH}/usage_events_stream/*.jsonl")
)

usage_stream_prepared = (
    usage_stream
    .withColumn("event_time", F.to_timestamp("timestamp"))
    .withColumn("source_file", F.input_file_name())
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("event_date", F.to_date("event_time"))
    .withWatermark(
        "event_time",
        f"{WATERMARK_MINUTES} minutes"
    )
    .dropDuplicatesWithinWatermark(["event_id"])
)

print("¿Es streaming?:", usage_stream_prepared.isStreaming)
usage_stream_prepared.printSchema()

¿Es streaming?: True
root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingest_ts: timestamp (nullable = false)
 |-- event_date: date (nullable = true)



## 5.2 Ejecución de Streaming → Bronze

Cada query utiliza su propio checkpoint.

El sink de archivos se ejecuta en modo `append`, mientras que el checkpoint conserva el estado necesario para evitar reprocesar archivos ya confirmados al volver a ejecutar la misma query.

In [65]:
detener_queries_activas()

bronze_stream_query = (
    usage_stream_prepared
    .writeStream
    .format("parquet")
    .queryName("bronze_usage_events")
    .outputMode("append")
    .option("path", EVENTS_BRONZE_PATH)
    .option("checkpointLocation", EVENTS_CHECKPOINT)
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .start()
)

bronze_stream_query.awaitTermination()
print("Streaming Bronze finalizado.")
print("Query activa:", bronze_stream_query.isActive)

No hay queries activas.
Streaming Bronze finalizado.
Query activa: False


## 5.3 Evidencia del progreso del stream

Se conserva el progreso reportado por Spark para poder demostrar micro-batches, cantidad de filas procesadas y estado de la query.

In [66]:
if bronze_stream_query.lastProgress:
    print(
        json.dumps(
            bronze_stream_query.lastProgress,
            indent=2,
            default=str
        )
    )
else:
    print("La query no reportó progreso disponible.")

{
  "id": "04c77abb-97c1-4695-a767-c7257205a25b",
  "runId": "c2aa7504-b8c8-4816-a7df-2e0a9d68dabe",
  "name": "bronze_usage_events",
  "timestamp": "2026-11-18T18:51:54.675Z",
  "batchId": 11,
  "batchDuration": 3654,
  "durationMs": {
    "addBatch": 3526,
    "commitOffsets": 50,
    "getBatch": 14,
    "latestOffset": 23,
    "queryPlanning": 15,
    "triggerExecution": 3654,
    "walCommit": 22
  },
  "eventTime": {
    "avg": "2025-08-01T17:28:24.333Z",
    "max": "2025-08-31T23:19:00.000Z",
    "min": "2025-07-03T00:26:00.000Z",
    "watermark": "2025-07-02T23:58:00.000Z"
  },
  "stateOperators": [
    {
      "operatorName": "dedupeWithinWatermark",
      "numRowsTotal": 43200,
      "numRowsUpdated": 3600,
      "numRowsRemoved": 0,
      "allUpdatesTimeMs": 107,
      "allRemovalsTimeMs": 6,
      "commitTimeMs": 295,
      "memoryUsedBytes": 10087584,
      "numRowsDroppedByWatermark": 0,
      "numShufflePartitions": 8,
      "numStateStoreInstances": 8,
      "customMetric

## 5.4 Validación física de Bronze streaming

La salida se lee nuevamente como un DataFrame estático para verificar que los datos realmente fueron escritos en Parquet.

También se comparan el total de filas y la cantidad de `event_id` distintos.

In [67]:
events_bronze = spark.read.parquet(EVENTS_BRONZE_PATH)

bronze_event_count = events_bronze.count()
bronze_distinct_event_ids = (
    events_bronze
    .select("event_id")
    .distinct()
    .count()
)

print("Eventos en Bronze:", bronze_event_count)
print("event_id distintos:", bronze_distinct_event_ids)
print(
    "Duplicados por event_id:",
    bronze_event_count - bronze_distinct_event_ids
)

(
    events_bronze
    .groupBy("schema_version")
    .count()
    .orderBy("schema_version")
    .show()
)

Eventos en Bronze: 43200
event_id distintos: 43200
Duplicados por event_id: 0
+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1|10800|
|             2|32400|
+--------------+-----+



### Interpretación del resultado

En E1 se habían observado 43.200 eventos y 43.200 `event_id` únicos.

La ejecución de Structured Streaming reprodujo ese resultado: Bronze contiene **43.200 eventos y 43.200 identificadores distintos**, por lo que la deduplicación no eliminó registros del dataset original.

El progreso del stream también mostró **0 registros descartados por watermark** con la tolerancia seleccionada.

La ausencia de duplicados en la muestra no vuelve innecesario el mecanismo de deduplicación. Su objetivo es proteger el pipeline frente a reenvíos o reprocesamientos posteriores, comportamiento que se verifica nuevamente mediante una prueba controlada en el Punto 15.

# 6 - Silver: conformado y enriquecimiento

Silver debe contener datos preparados para uso analítico.

Para E2 se conformará:

- el maestro `customers_orgs`;
- los eventos de utilización provenientes de Bronze.

Los eventos serán normalizados, sometidos a reglas de calidad y enriquecidos mediante un join con organizaciones.

## 6.1 Customers Silver

El maestro de organizaciones se normaliza sin alterar su significado de negocio.

Se aplican operaciones simples de limpieza sobre strings y se mantiene una única fila por `org_id`.

In [68]:
customers_silver = (
    spark.read.parquet(CUSTOMERS_BRONZE_PATH)
    .withColumn("org_name", F.trim("org_name"))
    .withColumn("industry", F.trim("industry"))
    .withColumn("hq_region", F.lower(F.trim("hq_region")))
    .withColumn("plan_tier", F.lower(F.trim("plan_tier")))
    .withColumn("lifecycle_stage", F.lower(F.trim("lifecycle_stage")))
    .dropDuplicates(["org_id"])
)

CUSTOMERS_SILVER_PATH = f"{SILVER_PATH}/customers_orgs"

(
    customers_silver
    .write
    .mode("overwrite")
    .parquet(CUSTOMERS_SILVER_PATH)
)

customers_silver_check = spark.read.parquet(CUSTOMERS_SILVER_PATH)
print("Customers Silver:", customers_silver_check.count())
customers_silver_check.show(5, truncate=False)

Customers Silver: 80
+------------+--------------+----------+----------+----------+-------------+-----------+---------+---------------+----------------+---------+--------------------------------------------------------------------------+--------------------------+-----------+
|org_id      |org_name      |industry  |hq_region |plan_tier |is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|source_file                                                               |ingest_ts                 |ingest_date|
+------------+--------------+----------+----------+----------+-------------+-----------+---------+---------------+----------------+---------+--------------------------------------------------------------------------+--------------------------+-----------+
|org_0lvsnujz|Nova Cloud 45 |Retail    |us-east   |standard  |false        |2025-05-19 |rep_e    |prospect       |event           |-1       |file:///content/cloud_provider_dataset/datalake/landing/customers_orgs

# 7 - Calidad y quarantine

La calidad se implementa como reglas verificables y no únicamente como una descripción.

Se utilizarán cuatro controles sobre los eventos:

- `event_id` debe existir;
- si existe `value`, debe existir `unit`;
- si `value` no es nulo, debe poder convertirse a número para las métricas que lo requieran;
- un costo menor a `-0.01` se marca como anomalía de costo.

Los tres primeros controles pueden impedir la promoción a Silver. El costo negativo se conserva para no perder información financiera, pero queda explícitamente marcado mediante un flag.

## 7.1 Normalización y reglas de calidad

Se transforma `value` a `value_num` mediante casting controlado y se genera un motivo de rechazo para cada registro que no pueda promocionarse.

In [69]:
events_quality = (
    events_bronze
    .withColumn("value_num", F.col("value").cast("double"))
    .withColumn(
        "event_id_missing",
        F.col("event_id").isNull()
        | (F.trim(F.col("event_id")) == "")
    )
    .withColumn(
        "unit_missing_for_value",
        F.col("value").isNotNull()
        & (
            F.col("unit").isNull()
            | (F.trim(F.col("unit")) == "")
        )
    )
    .withColumn(
        "value_cast_failed",
        F.col("value").isNotNull()
        & F.col("value_num").isNull()
    )
    .withColumn(
        "cost_anomaly_flag",
        F.coalesce(
            F.col("cost_usd_increment") < F.lit(-0.01),
            F.lit(False)
        )
    )
    .withColumn(
        "quality_reason",
        F.concat_ws(
            "|",
            F.when(F.col("event_id_missing"), F.lit("event_id_missing")),
            F.when(
                F.col("unit_missing_for_value"),
                F.lit("unit_missing_for_value")
            ),
            F.when(
                F.col("value_cast_failed"),
                F.lit("value_cast_failed")
            ),
        )
    )
    .withColumn(
        "is_valid_for_silver",
        F.length(F.col("quality_reason")) == 0
    )
)

## 7.2 Conteo de reglas

Los conteos permiten distinguir reglas que efectivamente encuentran problemas de aquellas que funcionan como controles preventivos.

In [70]:
quality_summary = (
    events_quality
    .agg(
        F.count("*").alias("total_eventos"),
        F.sum(F.col("event_id_missing").cast("int"))
        .alias("event_id_missing"),
        F.sum(F.col("unit_missing_for_value").cast("int"))
        .alias("unit_missing_for_value"),
        F.sum(F.col("value_cast_failed").cast("int"))
        .alias("value_cast_failed"),
        F.sum(F.col("cost_anomaly_flag").cast("int"))
        .alias("cost_anomaly_flag"),
        F.sum((~F.col("is_valid_for_silver")).cast("int"))
        .alias("quarantine_rows"),
    )
)

quality_summary.show(truncate=False)

+-------------+----------------+----------------------+-----------------+-----------------+---------------+
|total_eventos|event_id_missing|unit_missing_for_value|value_cast_failed|cost_anomaly_flag|quarantine_rows|
+-------------+----------------+----------------------+-----------------+-----------------+---------------+
|43200        |0               |2038                  |0                |211              |2038           |
+-------------+----------------+----------------------+-----------------+-----------------+---------------+



## 7.3 Generación de quarantine

Los registros que no superan las reglas de promoción se almacenan en Parquet junto con `quality_reason`.

Quarantine no modifica Landing y permite inspeccionar por qué un evento no fue promovido.

In [71]:
events_quarantine = (
    events_quality
    .filter(~F.col("is_valid_for_silver"))
)

EVENTS_QUARANTINE_PATH = f"{QUARANTINE_PATH}/usage_events"

(
    events_quarantine
    .write
    .mode("overwrite")
    .parquet(EVENTS_QUARANTINE_PATH)
)

quarantine_check = spark.read.parquet(EVENTS_QUARANTINE_PATH)
print("Registros en quarantine:", quarantine_check.count())

(
    quarantine_check
    .select(
        "event_id",
        "event_time",
        "metric",
        "value",
        "unit",
        "quality_reason",
        "source_file",
    )
    .show(20, truncate=False)
)

Registros en quarantine: 2038
+----------------+-------------------+----------------+-------+----+----------------------+--------------------------------------------------------------------------------------------------+
|event_id        |event_time         |metric          |value  |unit|quality_reason        |source_file                                                                                       |
+----------------+-------------------+----------------+-------+----+----------------------+--------------------------------------------------------------------------------------------------+
|evt_deqcgyhxggn4|2025-08-04 05:40:00|cpu_hours       |3.7042 |NULL|unit_missing_for_value|file:///content/cloud_provider_dataset/datalake/landing/usage_events_stream/events_part_0010.jsonl|
|evt_qetqdtuqsa19|2025-08-02 21:13:00|requests        |128.0  |NULL|unit_missing_for_value|file:///content/cloud_provider_dataset/datalake/landing/usage_events_stream/events_part_0115.jsonl|
|evt_lh61sa7s9z

# 8 - Features y eventos Silver

Los registros válidos se preparan para el análisis FinOps.

Se implementan cuatro features, superando el mínimo de tres solicitado:

- costo incremental utilizable para agregación;
- requests;
- tokens GenAI;
- carbono.

La lógica conserva cero cuando una métrica no aplica al evento, evitando convertir nulos estructurales en errores de calidad.

## 8.1 Construcción de features por evento

`requests` utiliza `value_num` solamente cuando `metric = requests`.

`genai_tokens` y `carbon_kg` conservan la semántica observada en schema v2 y se transforman a cero únicamente para facilitar agregaciones posteriores.

In [72]:
events_valid = events_quality.filter("is_valid_for_silver")

events_with_features = (
    events_valid
    .withColumn(
        "feature_cost_usd",
        F.coalesce(F.col("cost_usd_increment"), F.lit(0.0))
    )
    .withColumn(
        "feature_requests",
        F.when(
            F.col("metric") == "requests",
            F.coalesce(F.col("value_num"), F.lit(0.0))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "feature_genai_tokens",
        F.coalesce(F.col("genai_tokens"), F.lit(0)).cast("long")
    )
    .withColumn(
        "feature_carbon_kg",
        F.coalesce(F.col("carbon_kg"), F.lit(0.0))
    )
    .withColumn("usage_date", F.to_date("event_time"))
    .withColumn("usage_month", F.date_format("event_time", "yyyy-MM"))
)

events_with_features.select(
    "event_id",
    "metric",
    "value",
    "feature_cost_usd",
    "feature_requests",
    "feature_genai_tokens",
    "feature_carbon_kg",
).show(20, truncate=False)

+----------------+----------------+-------+----------------+----------------+--------------------+-----------------+
|event_id        |metric          |value  |feature_cost_usd|feature_requests|feature_genai_tokens|feature_carbon_kg|
+----------------+----------------+-------+----------------+----------------+--------------------+-----------------+
|evt_wpf3yzvu1lqw|cpu_hours       |1.7778 |0.0201          |0.0             |0                   |3.56E-4          |
|evt_55195m07uwwj|requests        |118.0  |9.4035          |118.0           |0                   |0.0236           |
|evt_0d222k2zeuj7|requests        |127.0  |6.1782          |127.0           |0                   |0.0254           |
|evt_dk2ovxujqvoe|requests        |121.0  |1.0972          |121.0           |0                   |0.0242           |
|evt_0ldj2ql5u07w|requests        |130.0  |8.9183          |130.0           |0                   |0.026            |
|evt_5yeeuoq25e7l|requests        |121.0  |11.7412         |121.

## 8.2 Join de enriquecimiento con organizaciones

El join es `stream-static` conceptualmente en la arquitectura, pero para construir Silver de forma reproducible luego de completar `availableNow`, se materializa a partir del Bronze ya escrito y el maestro Silver.

El enriquecimiento agrega contexto de organización sin modificar el grano del evento.

In [73]:
customer_dimension = (
    customers_silver_check
    .select(
        "org_id",
        "org_name",
        "industry",
        "hq_region",
        "plan_tier",
        "is_enterprise",
        "lifecycle_stage",
    )
)

events_silver = (
    events_with_features.alias("e")
    .join(
        F.broadcast(customer_dimension).alias("c"),
        on="org_id",
        how="left"
    )
)

orphan_silver = (
    events_silver
    .filter(F.col("org_name").isNull())
    .count()
)

print("Eventos válidos antes del join:", events_with_features.count())
print("Eventos Silver después del join:", events_silver.count())
print("Eventos sin organización maestra:", orphan_silver)

Eventos válidos antes del join: 41162
Eventos Silver después del join: 41162
Eventos sin organización maestra: 0


## 8.3 Escritura de Silver

Silver se almacena en Parquet y conserva `usage_month` como partición de baja cardinalidad para este dataset.

La decisión de particionamiento del mart Gold se evaluará posteriormente mediante una comparación específica.

In [74]:
EVENTS_SILVER_PATH = f"{SILVER_PATH}/usage_events"

(
    events_silver
    .write
    .mode("overwrite")
    .partitionBy("usage_month")
    .parquet(EVENTS_SILVER_PATH)
)

events_silver_check = spark.read.parquet(EVENTS_SILVER_PATH)

print("Eventos Silver persistidos:", events_silver_check.count())
events_silver_check.printSchema()

Eventos Silver persistidos: 41162
root
 |-- org_id: string (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- value_num: double (nullable = true)
 |-- event_id_missing: boolean (nullable = true)
 |-- unit_missing_for_value: boolean (nullable = true)
 |-- value_cast_failed: boolean (nullable = true)
 |-- cost_anomaly_flag: boolean (nullable = true)
 |-- quality_reas

# 9 - Componente analítico: anomalías de costo por servicio

La segunda evaluación solicita incorporar un componente analítico o de Machine Learning cuando corresponda, o una alternativa analítica equivalente.

Para este MVP se implementó una detección simple y explicable de spikes de costo mediante percentil 99 por servicio.

Se evita introducir un modelo más complejo cuando el objetivo actual es demostrar un pipeline end-to-end interpretable y reproducible.

## 9.1 Umbral por percentil 99

Cada servicio posee una escala de costos diferente. Por ese motivo el umbral se calcula por `service` y no sobre toda la población en conjunto.

El percentil 99 se obtiene directamente de los datos Silver.

In [75]:
service_thresholds = (
    events_silver_check
    .filter(F.col("cost_usd_increment").isNotNull())
    .groupBy("service")
    .agg(
        F.expr(
            "percentile_approx(cost_usd_increment, 0.99, 10000)"
        ).alias("cost_p99")
    )
)

service_thresholds.orderBy("service").show(truncate=False)

+----------+--------+
|service   |cost_p99|
+----------+--------+
|analytics |11.6259 |
|compute   |13.4861 |
|database  |8.3741  |
|genai     |19.7267 |
|networking|1.6749  |
|storage   |3.3395  |
+----------+--------+



## 9.2 Flag de spike analítico

Un evento se marca como `cost_spike_flag` cuando su costo supera el percentil 99 de su propio servicio.

Este flag es diferente de `cost_anomaly_flag`, que representa la regla de calidad para costos menores a `-0.01`.

In [76]:
events_silver_analytic = (
    events_silver_check
    .join(service_thresholds, "service", "left")
    .withColumn(
        "cost_spike_flag",
        F.coalesce(
            F.col("cost_usd_increment") > F.col("cost_p99"),
            F.lit(False)
        )
    )
)

(
    events_silver_analytic
    .groupBy("service")
    .agg(
        F.count("*").alias("eventos"),
        F.sum(F.col("cost_spike_flag").cast("int"))
        .alias("spikes_p99"),
        F.max("cost_usd_increment").alias("max_cost"),
        F.first("cost_p99").alias("cost_p99"),
    )
    .orderBy("service")
    .show(truncate=False)
)

+----------+-------+----------+--------+--------+
|service   |eventos|spikes_p99|max_cost|cost_p99|
+----------+-------+----------+--------+--------+
|analytics |4389   |43        |249.3861|11.6259 |
|compute   |11882  |118       |302.3459|13.4861 |
|database  |7068   |71        |149.9091|8.3741  |
|genai     |3961   |39        |317.4308|19.7267 |
|networking|6607   |67        |29.6319 |1.6749  |
|storage   |7255   |73        |62.9922 |3.3395  |
+----------+-------+----------+--------+--------+



## 9.3 Interpretación del componente analítico

El percentil 99 no afirma que todo evento marcado sea un error.

El flag identifica observaciones inusualmente altas respecto del comportamiento del mismo servicio y permite priorizar su revisión.

La principal ventaja para E2 es que el método es simple, trazable y puede explicarse a partir de los datos utilizados. En una etapa posterior podría compararse con otros métodos de anomalías si existiera valor de negocio para hacerlo.

# 10 - Gold: org_daily_usage_by_service

El mart FinOps requerido posee el siguiente grano:

`org_id + usage_date + service`

Cada fila representa el comportamiento diario de un servicio para una organización.

Se incluirán métricas de costo, requests, GenAI, carbono y evidencia de anomalías.

## 10.1 Construcción del mart

La agregación parte exclusivamente de Silver para respetar la promoción entre capas.

In [77]:
gold_daily = (
    events_silver_analytic
    .groupBy(
        "org_id",
        "usage_date",
        "service",
    )
    .agg(
        F.round(F.sum("feature_cost_usd"), 6)
        .alias("daily_cost_usd"),
        F.round(F.sum("feature_requests"), 0)
        .cast("long")
        .alias("requests"),
        F.sum("feature_genai_tokens")
        .cast("long")
        .alias("genai_tokens"),
        F.round(F.sum("feature_carbon_kg"), 6)
        .alias("carbon_kg"),
        F.count("*").alias("event_count"),
        F.sum(F.col("cost_anomaly_flag").cast("int"))
        .cast("int")
        .alias("negative_cost_events"),
        F.sum(F.col("cost_spike_flag").cast("int"))
        .cast("int")
        .alias("cost_spike_events"),
    )
    .withColumn(
        "usage_month",
        F.date_format("usage_date", "yyyy-MM")
    )
)

print("Filas Gold:", gold_daily.count())
gold_daily.orderBy("usage_date", "org_id", "service").show(20, truncate=False)

Filas Gold: 11050
+------------+----------+----------+--------------+--------+------------+---------+-----------+--------------------+-----------------+-----------+
|org_id      |usage_date|service   |daily_cost_usd|requests|genai_tokens|carbon_kg|event_count|negative_cost_events|cost_spike_events|usage_month|
+------------+----------+----------+--------------+--------+------------+---------+-----------+--------------------+-----------------+-----------+
|org_0lvsnujz|2025-07-03|analytics |8.8739        |116     |0           |0.0      |3          |0                   |0                |2025-07    |
|org_0lvsnujz|2025-07-03|database  |0.5723        |0       |0           |0.0      |2          |0                   |0                |2025-07    |
|org_0lvsnujz|2025-07-03|storage   |5.5542        |255     |0           |0.0      |3          |0                   |0                |2025-07    |
|org_0lzjjege|2025-07-03|compute   |12.6064       |120     |0           |0.0      |3          |0    

## 10.2 Validación del grano

La combinación `org_id + usage_date + service` debe ser única.

Esta validación permite comprobar que el mart respeta el grano declarado y evita duplicados lógicos antes de cargar Cassandra.

In [78]:
gold_duplicate_keys = (
    gold_daily
    .groupBy("org_id", "usage_date", "service")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves Gold duplicadas:", gold_duplicate_keys)
assert gold_duplicate_keys == 0, "Gold contiene claves duplicadas."

Claves Gold duplicadas: 0


# 11 - Decisión de particionamiento

El feedback de E1 solicita que el particionamiento deje de ser solamente conceptual.

Para Gold se compararán alternativas utilizando el mismo DataFrame y midiendo:

- cantidad de archivos Parquet;
- tamaño total;
- tamaño promedio por archivo.

Las alternativas serán:

- sin partición;
- partición por mes;
- partición por fecha;
- partición por fecha y servicio.

La consigna requiere una estrategia particionada; la alternativa sin partición se conserva únicamente como referencia de comparación.

## 11.1 Escritura de alternativas

Las pruebas se almacenan en una ruta temporal separada de Gold definitivo.

In [79]:
PARTITION_TEST_ROOT = f"{TEST_PATH}/gold_partitioning"

if os.path.exists(PARTITION_TEST_ROOT):
    shutil.rmtree(PARTITION_TEST_ROOT)

partition_candidates = {
    "sin_particion": [],
    "por_mes": ["usage_month"],
    "por_fecha": ["usage_date"],
    "por_fecha_servicio": ["usage_date", "service"],
}

for name, partition_cols in partition_candidates.items():
    path = f"{PARTITION_TEST_ROOT}/{name}"
    writer = gold_daily.write.mode("overwrite")

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    writer.parquet(path)

print("Alternativas escritas.")

Alternativas escritas.


## 11.2 Medición de archivos generados

En un dataset pequeño, una cantidad excesiva de particiones puede generar muchos archivos de pocos kilobytes.

La medición permite comparar el beneficio de segmentar temporalmente con el costo de producir archivos pequeños.

In [80]:
def parquet_file_stats(path):
    parquet_files = [
        p for p in Path(path).rglob("*.parquet")
        if p.is_file()
    ]

    sizes = [p.stat().st_size for p in parquet_files]
    total_bytes = sum(sizes)

    return (
        len(parquet_files),
        total_bytes,
        (total_bytes / len(parquet_files)) if parquet_files else 0.0,
    )

partition_measurements = []

for name in partition_candidates:
    path = f"{PARTITION_TEST_ROOT}/{name}"
    files_count, total_bytes, avg_bytes = parquet_file_stats(path)
    partition_measurements.append(
        (name, files_count, total_bytes, round(avg_bytes, 2))
    )

partition_evidence = spark.createDataFrame(
    partition_measurements,
    [
        "alternativa",
        "archivos_parquet",
        "bytes_totales",
        "bytes_promedio_archivo",
    ]
)

partition_evidence.show(truncate=False)

+------------------+----------------+-------------+----------------------+
|alternativa       |archivos_parquet|bytes_totales|bytes_promedio_archivo|
+------------------+----------------+-------------+----------------------+
|sin_particion     |1               |203367       |203367.0              |
|por_mes           |2               |208623       |104311.5              |
|por_fecha         |60              |478246       |7970.77               |
|por_fecha_servicio|360             |1419145      |3942.07               |
+------------------+----------------+-------------+----------------------+



## 11.3 Decisión para Gold

Para el MVP se seleccionó **partición por `usage_month`**.

La comparación física produjo los siguientes resultados:

- sin partición: 1 archivo Parquet;
- por mes: 2 archivos;
- por fecha: 60 archivos;
- por fecha y servicio: 360 archivos.

La partición mensual mantiene una dimensión temporal útil sin producir la fragmentación observada en las alternativas diarias.

El mart Gold contiene **11.050 registros**, distribuidos en:

- julio de 2025: 5.347 registros;
- agosto de 2025: 5.703 registros.

La decisión queda respaldada por la medición realizada sobre el dataset disponible. En un volumen productivo debería revisarse nuevamente considerando el tamaño real de archivos, frecuencia de consultas y patrón de crecimiento.

In [81]:
GOLD_DAILY_PATH = f"{GOLD_PATH}/org_daily_usage_by_service"

(
    gold_daily
    .write
    .mode("overwrite")
    .partitionBy("usage_month")
    .parquet(GOLD_DAILY_PATH)
)

gold_check = spark.read.parquet(GOLD_DAILY_PATH)

print("Gold persistido:", gold_check.count())
print("Particiones por month:")
(
    gold_check
    .groupBy("usage_month")
    .count()
    .orderBy("usage_month")
    .show()
)

Gold persistido: 11050
Particiones por month:
+-----------+-----+
|usage_month|count|
+-----------+-----+
|    2025-07| 5347|
|    2025-08| 5703|
+-----------+-----+



# 12 - Retención y arquitectura implementada

La política de retención había quedado abierta en E1.

Para evitar inventar una política productiva sin requisitos regulatorios o comerciales, se separa la decisión del MVP académico de una eventual política productiva.

## 12.1 Política de retención del MVP

Para esta implementación se adopta la siguiente política concreta:

| Capa | Retención durante el proyecto |
|---|---|
| Landing | Conservar el dataset original durante todo el proyecto como fuente reproducible |
| Bronze | Regenerable desde Landing; se conserva durante la ejecución y las demostraciones |
| Silver | Regenerable desde Bronze; se conserva durante la ejecución y las demostraciones |
| Gold | Regenerable desde Silver; se conserva durante la ejecución y para carga a serving |
| Quarantine | Conservar durante el proyecto para auditoría de reglas de calidad |
| Cassandra/AstraDB E2 | Sin TTL durante el período académico para no eliminar evidencia antes de la instancia final |

Google Colab utiliza almacenamiento temporal. La reproducibilidad depende por lo tanto de conservar Landing fuera de la sesión y de mantener código, configuración e instrucciones en GitHub.

Una retención productiva definitiva permanece fuera del alcance de E2 porque no existe un requisito de negocio, cumplimiento o costo que permita fijarla responsablemente.

## 12.2 Arquitectura v2 implementada

La arquitectura de E1 se mantiene conceptualmente, pero E2 reemplaza componentes aspiracionales por componentes ejecutados.

```text
FUENTES
  ├── CSV batch
  └── usage_events_stream/*.jsonl
            │
            ▼
         LANDING
            │
      ┌─────┴─────────────┐
      │                   │
      ▼                   ▼
 PySpark batch      Structured Streaming
 esquema explícito  esquema + watermark
 metadata           dedupe + checkpoint
      │                   │
      └───────┬───────────┘
              ▼
            BRONZE
           Parquet
              │
              ▼
            SILVER
   calidad + quarantine
   features + join maestro
              │
              ▼
             GOLD
 org_daily_usage_by_service
              │
              ▼
      Cassandra / AstraDB
              │
              ▼
        Consultas CQL
```

Capacidades transversales implementadas o evidenciadas: metadata técnica, calidad, linaje, checkpoints, idempotencia, seguridad de credenciales y registro de decisiones.

# 13 - Serving en Cassandra / AstraDB

La capa de serving se implementó en DataStax AstraDB siguiendo el enfoque query-first trabajado en clase.

Para E2 se necesita una tabla que responda eficientemente consultas temporales de costo y requests para una organización y servicio.

El acceso a AstraDB requiere una intervención manual porque las credenciales no deben almacenarse dentro del notebook ni del repositorio.

## 13.1 Diseño query-first

La consulta prioritaria implementada en E2 es:

**Obtener costos y requests diarios de un servicio para una organización en un rango de fechas.**

La tabla utilizará:

- partition key: `(org_id, service)`;
- clustering column: `usage_date`;
- orden descendente por fecha.

Esto permite que una consulta habitual lea una única partición lógica y recorra las fechas en orden.

El modelo no pretende resolver todavía todas las consultas de la instancia final. Las consultas adicionales podrán requerir nuevas tablas query-first, respetando el principio de Cassandra de modelar según las consultas.

## 13.2 Paso manual en AstraDB

Antes de ejecutar la siguiente sección:

- ingresar al portal de DataStax AstraDB;
- crear o seleccionar una base para el proyecto;
- crear el keyspace `cloud_provider_analytics` desde el portal;
- generar un **Application Token** para el laboratorio;
- descargar el **Secure Connect Bundle (SCB)** de la base;
- conservar el token fuera de GitHub;
- no descomprimir el SCB.

El material de clase indica que en AstraDB el keyspace se administra desde el servicio y no mediante `CREATE KEYSPACE` en CQL.

La próxima celda se detendrá esperando que se cargue el Secure Connect Bundle.

In [82]:
!pip -q install "cassandra-driver>=3.28"

In [83]:
from google.colab import files
from getpass import getpass

print("Cargá únicamente el Secure Connect Bundle (.zip) descargado de AstraDB.")
scb_uploaded = files.upload()

scb_candidates = [
    name for name in scb_uploaded.keys()
    if name.lower().endswith(".zip")
]

if len(scb_candidates) != 1:
    raise ValueError("Se esperaba un único Secure Connect Bundle ZIP.")

ASTRA_SCB = f"/content/{scb_candidates[0]}"
ASTRA_TOKEN = getpass("Pegá el Application Token de AstraDB: ")
ASTRA_KEYSPACE = "cloud_provider_analytics"

if not ASTRA_TOKEN.startswith("AstraCS:"):
    print("Advertencia: el token no comienza con el prefijo AstraCS esperado.")

print("SCB cargado en memoria temporal.")
print("Keyspace esperado:", ASTRA_KEYSPACE)

Cargá únicamente el Secure Connect Bundle (.zip) descargado de AstraDB.


Saving secure-connect-cloud-provider-analytics.zip to secure-connect-cloud-provider-analytics (1).zip
Pegá el Application Token de AstraDB: ··········
SCB cargado en memoria temporal.
Keyspace esperado: cloud_provider_analytics


## 13.3 Conexión segura

El token se mantiene únicamente en memoria durante la sesión de Colab.

El Secure Connect Bundle contiene la información TLS necesaria para alcanzar la base sin exponer direcciones o certificados dentro del código.

In [84]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

cluster = Cluster(
    cloud={"secure_connect_bundle": ASTRA_SCB},
    auth_provider=PlainTextAuthProvider(
        "token",
        ASTRA_TOKEN
    )
)

session = cluster.connect(ASTRA_KEYSPACE)

row = session.execute(
    "SELECT release_version FROM system.local"
).one()

print("Conectado a AstraDB.")
print("Versión CQL del servicio:", row.release_version)

Conectado a AstraDB.
Versión CQL del servicio: 4.0.11.0-6819aa9c7f6c


## 13.4 Script CQL de E2

Se crea una tabla query-first para `org_daily_usage_by_service`.

No se utiliza TTL durante el período académico para evitar que la evidencia de E2 desaparezca antes de la entrega final.

In [85]:
CREATE_GOLD_TABLE_CQL = """
CREATE TABLE IF NOT EXISTS org_daily_usage_by_service (
    org_id text,
    service text,
    usage_date date,
    daily_cost_usd double,
    requests bigint,
    genai_tokens bigint,
    carbon_kg double,
    event_count int,
    negative_cost_events int,
    cost_spike_events int,
    PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)
"""

session.execute(CREATE_GOLD_TABLE_CQL)
print("Tabla creada o ya existente.")

cql_file = f"{CQL_PATH}/02_evaluacion_2.cql"
with open(cql_file, "w", encoding="utf-8") as f:
    f.write("USE cloud_provider_analytics;\n\n")
    f.write(CREATE_GOLD_TABLE_CQL.strip())
    f.write(";\n")

print("Script CQL generado:", cql_file)

Tabla creada o ya existente.
Script CQL generado: /content/cloud_provider_e2/cql/02_evaluacion_2.cql


## 13.5 Carga del mart desde Spark

Los datos provienen del DataFrame Gold generado por Spark.

Se utilizan sentencias preparadas del driver de Cassandra. Como la clave primaria está formada por `(org_id, service, usage_date)`, volver a insertar la misma fila actúa como un upsert y no crea un duplicado lógico.

In [86]:
from cassandra.concurrent import execute_concurrent_with_args

insert_gold = session.prepare("""
INSERT INTO org_daily_usage_by_service (
    org_id,
    service,
    usage_date,
    daily_cost_usd,
    requests,
    genai_tokens,
    carbon_kg,
    event_count,
    negative_cost_events,
    cost_spike_events
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

rows_for_astra = [
    (
        row.org_id,
        row.service,
        row.usage_date,
        float(row.daily_cost_usd or 0.0),
        int(row.requests or 0),
        int(row.genai_tokens or 0),
        float(row.carbon_kg or 0.0),
        int(row.event_count or 0),
        int(row.negative_cost_events or 0),
        int(row.cost_spike_events or 0),
    )
    for row in (
        gold_check
        .select(
            "org_id",
            "service",
            "usage_date",
            "daily_cost_usd",
            "requests",
            "genai_tokens",
            "carbon_kg",
            "event_count",
            "negative_cost_events",
            "cost_spike_events",
        )
        .toLocalIterator()
    )
]

print("Filas preparadas para AstraDB:", len(rows_for_astra))

results = execute_concurrent_with_args(
    session,
    insert_gold,
    rows_for_astra,
    concurrency=50,
    raise_on_first_error=True,
)

successful = sum(1 for ok, _ in results if ok)
print("Upserts completados:", successful)

Filas preparadas para AstraDB: 11050
Upserts completados: 11050


# 14 - Consultas CQL y evidencia de serving

Se ejecutaron dos consultas sobre la tabla query-first implementada.

La primera recupera un rango de fechas para una organización y servicio. La segunda recupera los últimos registros de esa misma partición.

Las consultas se eligen de manera automática a partir de una combinación `org_id + service` que exista en Gold para garantizar una demostración reproducible.

## 14.1 Selección de una partición de ejemplo

Se utiliza una partición real con actividad y se obtienen sus fechas mínima y máxima.

In [87]:
sample_partition = (
    gold_check
    .groupBy("org_id", "service")
    .agg(
        F.count("*").alias("days"),
        F.min("usage_date").alias("min_date"),
        F.max("usage_date").alias("max_date"),
        F.sum("daily_cost_usd").alias("total_cost"),
    )
    .orderBy(F.desc("days"), F.desc("total_cost"))
    .first()
)

SAMPLE_ORG = sample_partition.org_id
SAMPLE_SERVICE = sample_partition.service
SAMPLE_MIN_DATE = sample_partition.min_date
SAMPLE_MAX_DATE = sample_partition.max_date

print("org_id:", SAMPLE_ORG)
print("service:", SAMPLE_SERVICE)
print("desde:", SAMPLE_MIN_DATE)
print("hasta:", SAMPLE_MAX_DATE)

org_id: org_ktakpuxq
service: compute
desde: 2025-07-03
hasta: 2025-08-31


## 14.2 Consulta 1 - Rango de fechas

CQL utilizado:

```sql
SELECT usage_date, daily_cost_usd, requests
FROM org_daily_usage_by_service
WHERE org_id = ?
  AND service = ?
  AND usage_date >= ?
  AND usage_date <= ?;
```

La consulta coincide con la partition key y utiliza `usage_date` como rango de clustering.

In [88]:
query_range = session.prepare("""
SELECT usage_date, daily_cost_usd, requests
FROM org_daily_usage_by_service
WHERE org_id = ?
  AND service = ?
  AND usage_date >= ?
  AND usage_date <= ?
""")

rows_q1 = list(session.execute(
    query_range,
    (
        SAMPLE_ORG,
        SAMPLE_SERVICE,
        SAMPLE_MIN_DATE,
        SAMPLE_MAX_DATE,
    )
))

print("Filas Q1:", len(rows_q1))
for row in rows_q1[:15]:
    print(row)

Filas Q1: 60
Row(usage_date=Date(20331), daily_cost_usd=27.9866, requests=344)
Row(usage_date=Date(20330), daily_cost_usd=21.9982, requests=225)
Row(usage_date=Date(20329), daily_cost_usd=10.2654, requests=123)
Row(usage_date=Date(20328), daily_cost_usd=49.3702, requests=469)
Row(usage_date=Date(20327), daily_cost_usd=65.1642, requests=715)
Row(usage_date=Date(20326), daily_cost_usd=69.17, requests=828)
Row(usage_date=Date(20325), daily_cost_usd=28.2485, requests=338)
Row(usage_date=Date(20324), daily_cost_usd=32.8164, requests=132)
Row(usage_date=Date(20323), daily_cost_usd=44.7278, requests=481)
Row(usage_date=Date(20322), daily_cost_usd=69.8615, requests=875)
Row(usage_date=Date(20321), daily_cost_usd=72.2359, requests=956)
Row(usage_date=Date(20320), daily_cost_usd=53.3694, requests=600)
Row(usage_date=Date(20319), daily_cost_usd=46.9234, requests=519)
Row(usage_date=Date(20318), daily_cost_usd=26.8727, requests=514)
Row(usage_date=Date(20317), daily_cost_usd=31.7552, requests=379)

## 14.3 Consulta 2 - Últimos días disponibles

CQL utilizado:

```sql
SELECT usage_date, daily_cost_usd, requests,
       genai_tokens, carbon_kg
FROM org_daily_usage_by_service
WHERE org_id = ?
  AND service = ?
LIMIT 10;
```

El `CLUSTERING ORDER BY (usage_date DESC)` permite obtener primero los días más recientes de la partición.

In [89]:
query_latest = session.prepare("""
SELECT usage_date,
       daily_cost_usd,
       requests,
       genai_tokens,
       carbon_kg
FROM org_daily_usage_by_service
WHERE org_id = ?
  AND service = ?
LIMIT 10
""")

rows_q2 = list(session.execute(
    query_latest,
    (SAMPLE_ORG, SAMPLE_SERVICE)
))

print("Filas Q2:", len(rows_q2))
for row in rows_q2:
    print(row)

Filas Q2: 10
Row(usage_date=Date(20331), daily_cost_usd=27.9866, requests=344, genai_tokens=0, carbon_kg=0.071488)
Row(usage_date=Date(20330), daily_cost_usd=21.9982, requests=225, genai_tokens=0, carbon_kg=0.052582)
Row(usage_date=Date(20329), daily_cost_usd=10.2654, requests=123, genai_tokens=0, carbon_kg=0.025916)
Row(usage_date=Date(20328), daily_cost_usd=49.3702, requests=469, genai_tokens=0, carbon_kg=0.109133)
Row(usage_date=Date(20327), daily_cost_usd=65.1642, requests=715, genai_tokens=0, carbon_kg=0.144818)
Row(usage_date=Date(20326), daily_cost_usd=69.17, requests=828, genai_tokens=0, carbon_kg=0.176972)
Row(usage_date=Date(20325), daily_cost_usd=28.2485, requests=338, genai_tokens=0, carbon_kg=0.073033)
Row(usage_date=Date(20324), daily_cost_usd=32.8164, requests=132, genai_tokens=0, carbon_kg=0.036051)
Row(usage_date=Date(20323), daily_cost_usd=44.7278, requests=481, genai_tokens=0, carbon_kg=0.104989)
Row(usage_date=Date(20322), daily_cost_usd=69.8615, requests=875, genai

## 14.4 Exportación del script CQL completo

Se consolida en un único archivo el DDL de la tabla query-first y las dos consultas CQL ejecutadas durante E2.

El archivo se conserva como artefacto reproducible para incorporarlo al repositorio.

In [98]:
QUERY_1_CQL = f"""
SELECT usage_date, daily_cost_usd, requests
FROM org_daily_usage_by_service
WHERE org_id = '{SAMPLE_ORG}'
  AND service = '{SAMPLE_SERVICE}'
  AND usage_date >= '{SAMPLE_MIN_DATE}'
  AND usage_date <= '{SAMPLE_MAX_DATE}';
"""

QUERY_2_CQL = f"""
SELECT usage_date,
       daily_cost_usd,
       requests,
       genai_tokens,
       carbon_kg
FROM org_daily_usage_by_service
WHERE org_id = '{SAMPLE_ORG}'
  AND service = '{SAMPLE_SERVICE}'
LIMIT 10;
"""

with open(cql_file, "w", encoding="utf-8") as f:
    f.write("-- Cloud Provider Analytics - Evaluación 2\n")
    f.write("-- Keyspace utilizado en AstraDB\n\n")

    f.write("USE cloud_provider_analytics;\n\n")

    f.write("-- Tabla query-first del mart Gold\n")
    f.write(CREATE_GOLD_TABLE_CQL.strip())
    f.write(";\n\n")

    f.write(
        "-- Consulta 1: costos y requests diarios "
        "por organización y servicio en un rango\n"
    )
    f.write(QUERY_1_CQL.strip())
    f.write("\n\n")

    f.write(
        "-- Consulta 2: últimos 10 registros "
        "de una partición organización + servicio\n"
    )
    f.write(QUERY_2_CQL.strip())
    f.write("\n")

print("Script CQL actualizado:", cql_file)

with open(cql_file, "r", encoding="utf-8") as f:
    print("\nContenido generado:\n")
    print(f.read())

Script CQL actualizado: /content/cloud_provider_e2/cql/02_evaluacion_2.cql

Contenido generado:

-- Cloud Provider Analytics - Evaluación 2
-- Keyspace utilizado en AstraDB

USE cloud_provider_analytics;

-- Tabla query-first del mart Gold
CREATE TABLE IF NOT EXISTS org_daily_usage_by_service (
    org_id text,
    service text,
    usage_date date,
    daily_cost_usd double,
    requests bigint,
    genai_tokens bigint,
    carbon_kg double,
    event_count int,
    negative_cost_events int,
    cost_spike_events int,
    PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC);

-- Consulta 1: costos y requests diarios por organización y servicio en un rango
SELECT usage_date, daily_cost_usd, requests
FROM org_daily_usage_by_service
WHERE org_id = 'org_ktakpuxq'
  AND service = 'compute'
  AND usage_date >= '2025-07-03'
  AND usage_date <= '2025-08-31';

-- Consulta 2: últimos 10 registros de una partición organización + servicio
SELECT usage_date,
   

## 14.5 Alcance del modelo de serving de E2

La tabla implementada responde correctamente consultas temporales por organización y servicio.

No se utilizó `ALLOW FILTERING` para forzar consultas incompatibles con la clave primaria.

La instancia final requerirá nuevas tablas query-first para responder consultas con otros patrones, como Top-N de servicios por organización. Esa necesidad se incorporará al backlog final en lugar de deformar la tabla actual.

# 15 - Idempotencia y re-ejecución

La idempotencia debe demostrarse mediante ejecución y conteos.

Se comprobaron tres mecanismos distintos:

- batch con escritura `overwrite`;
- streaming mediante checkpoint;
- Cassandra mediante upsert sobre la misma primary key.

## 15.1 Re-ejecución batch

Volver a escribir Customers Bronze no debe sumar nuevas filas porque la salida se reemplaza de forma determinística.

In [90]:
customers_count_before_rerun = (
    spark.read
    .parquet(CUSTOMERS_BRONZE_PATH)
    .count()
)

(
    customers_bronze
    .write
    .mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(CUSTOMERS_BRONZE_PATH)
)

# Se vuelve a crear el DataFrame luego del overwrite
# para evitar referencias a archivos Parquet anteriores.
customers_bronze_check = spark.read.parquet(
    CUSTOMERS_BRONZE_PATH
)

customers_count_after_rerun = (
    customers_bronze_check.count()
)

print("Antes:", customers_count_before_rerun)
print("Después:", customers_count_after_rerun)
print(
    "¿Idempotente?:",
    customers_count_before_rerun
    == customers_count_after_rerun
)

Antes: 80
Después: 80
¿Idempotente?: True


## 15.2 Re-ejecución del streaming con el mismo checkpoint

La query se vuelve a iniciar utilizando exactamente el mismo directorio de entrada, salida y checkpoint.

Como los 120 archivos ya fueron confirmados en el checkpoint, Spark no debería volver a agregarlos a Bronze.

In [91]:
stream_count_before_rerun = spark.read.parquet(
    EVENTS_BRONZE_PATH
).count()

usage_stream_rerun = (
    spark.readStream
    .schema(usage_event_schema)
    .option("maxFilesPerTrigger", 10)
    .json(f"{LANDING_PATH}/usage_events_stream/*.jsonl")
    .withColumn("event_time", F.to_timestamp("timestamp"))
    .withColumn("source_file", F.input_file_name())
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("event_date", F.to_date("event_time"))
    .withWatermark(
        "event_time",
        f"{WATERMARK_MINUTES} minutes"
    )
    .dropDuplicatesWithinWatermark(["event_id"])
)

rerun_query = (
    usage_stream_rerun
    .writeStream
    .format("parquet")
    .queryName("bronze_usage_events_rerun")
    .outputMode("append")
    .option("path", EVENTS_BRONZE_PATH)
    .option("checkpointLocation", EVENTS_CHECKPOINT)
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .start()
)

rerun_query.awaitTermination()

stream_count_after_rerun = spark.read.parquet(
    EVENTS_BRONZE_PATH
).count()

print("Antes:", stream_count_before_rerun)
print("Después:", stream_count_after_rerun)
print(
    "¿Checkpoint evitó duplicación?:",
    stream_count_before_rerun == stream_count_after_rerun
)

Antes: 43200
Después: 43200
¿Checkpoint evitó duplicación?: True


## 15.3 Prueba controlada de deduplicación

La fuente real no contiene `event_id` duplicados. Para demostrar el comportamiento de la deduplicación se crea una prueba aislada sin modificar Landing.

Se copia dos veces el mismo archivo JSONL a una carpeta de prueba. El origen tendrá aproximadamente el doble de filas, pero el stream con deduplicación deberá conservar una sola ocurrencia por `event_id`.

In [92]:
DEDUPE_TEST_INPUT = f"{TEST_PATH}/dedupe_input"
DEDUPE_TEST_OUTPUT = f"{TEST_PATH}/dedupe_output"
DEDUPE_TEST_CHECKPOINT = f"{CHECKPOINT_PATH}/dedupe_test"

for path in [
    DEDUPE_TEST_INPUT,
    DEDUPE_TEST_OUTPUT,
    DEDUPE_TEST_CHECKPOINT,
]:
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

source_test_file = event_files[0]
shutil.copy2(source_test_file, f"{DEDUPE_TEST_INPUT}/copy_a.jsonl")
shutil.copy2(source_test_file, f"{DEDUPE_TEST_INPUT}/copy_b.jsonl")

raw_test = (
    spark.read
    .schema(usage_event_schema)
    .json(f"{DEDUPE_TEST_INPUT}/*.jsonl")
)

raw_test_count = raw_test.count()
raw_test_distinct = raw_test.select("event_id").distinct().count()

print("Filas de prueba antes de dedupe:", raw_test_count)
print("event_id distintos esperados:", raw_test_distinct)

Filas de prueba antes de dedupe: 720
event_id distintos esperados: 360


In [93]:
dedupe_test_stream = (
    spark.readStream
    .schema(usage_event_schema)
    .json(f"{DEDUPE_TEST_INPUT}/*.jsonl")
    .withColumn("event_time", F.to_timestamp("timestamp"))
    .withWatermark(
        "event_time",
        f"{WATERMARK_MINUTES} minutes"
    )
    .dropDuplicatesWithinWatermark(["event_id"])
)

dedupe_test_query = (
    dedupe_test_stream
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", DEDUPE_TEST_OUTPUT)
    .option("checkpointLocation", DEDUPE_TEST_CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

dedupe_test_query.awaitTermination()

dedupe_test_result = spark.read.parquet(DEDUPE_TEST_OUTPUT)
dedupe_test_count = dedupe_test_result.count()

print("Filas luego de dedupe:", dedupe_test_count)
print(
    "¿Coincide con event_id distintos?:",
    dedupe_test_count == raw_test_distinct
)

Filas luego de dedupe: 360
¿Coincide con event_id distintos?: True


## 15.4 Idempotencia en AstraDB

Cassandra utiliza upsert sobre la primary key. Volver a insertar las mismas filas no debe crear nuevas filas para una misma combinación `(org_id, service, usage_date)`.

Se medirá una partición antes y después de repetir la carga.

In [94]:
count_partition = session.prepare("""
SELECT COUNT(*) AS total
FROM org_daily_usage_by_service
WHERE org_id = ? AND service = ?
""")

astra_count_before = session.execute(
    count_partition,
    (SAMPLE_ORG, SAMPLE_SERVICE)
).one().total

execute_concurrent_with_args(
    session,
    insert_gold,
    rows_for_astra,
    concurrency=50,
    raise_on_first_error=True,
)

astra_count_after = session.execute(
    count_partition,
    (SAMPLE_ORG, SAMPLE_SERVICE)
).one().total

print("Partición antes:", astra_count_before)
print("Partición después:", astra_count_after)
print(
    "¿Upsert idempotente?:",
    astra_count_before == astra_count_after
)

Partición antes: 60
Partición después: 60
¿Upsert idempotente?: True


# 16 - Gobierno, metadata, linaje y seguridad

E2 incorpora controles iniciales de gobierno que puedan verificarse directamente en la solución implementada.

## 16.1 Linaje implementado

El linaje principal puede resumirse de la siguiente manera:

| Producto | Origen inmediato | Origen inicial | Transformaciones principales |
|---|---|---|---|
| Customers Bronze | CSV | `customers_orgs.csv` | esquema, tipos, metadata, dedupe |
| Users Bronze | CSV | `users.csv` | esquema, tipos, metadata, dedupe |
| Billing Bronze | CSV | `billing_monthly.csv` | esquema, tipos, metadata, dedupe |
| Usage Bronze | JSONL streaming | `usage_events_stream/*.jsonl` | event time, watermark, dedupe, metadata |
| Customers Silver | Customers Bronze | `customers_orgs.csv` | normalización |
| Usage Silver | Usage Bronze + Customers Silver | JSONL + customers | calidad, casting, features, join |
| Quarantine | Usage Bronze | JSONL | reglas de calidad y motivo de rechazo |
| Gold FinOps | Usage Silver | fuentes anteriores | agregación diaria por org y servicio |
| AstraDB | Gold FinOps | fuentes anteriores | upsert por clave query-first |

## 16.2 Responsabilidad y acceso

El proyecto se desarrolla de forma individual.

**Responsable:** Nicolás Granata.

Controles aplicados:

- Landing se trata como inmutable;
- las credenciales de AstraDB no se almacenan en GitHub;
- el token se solicita en tiempo de ejecución mediante entrada oculta;
- el Secure Connect Bundle permanece únicamente en la sesión temporal de Colab;
- los datos rechazados conservan un motivo de quarantine;
- los checkpoints se mantienen separados por query;
- las claves naturales y primary keys documentan la identidad de cada registro.

## 16.3 Metadata técnica verificable

Las columnas técnicas principales implementadas son:

- `source_file` para rastrear el archivo de origen;
- `ingest_ts` para registrar cuándo se incorporó el dato;
- `schema_version` para conservar la evolución v1/v2 de los eventos;
- `quality_reason` para explicar por qué un registro fue enviado a quarantine;
- `cost_anomaly_flag` y `cost_spike_flag` para distinguir controles de calidad y analítica.

# 17 - Pruebas, reproducibilidad y esfuerzo

El notebook se diseñó para poder ejecutarse desde un entorno limpio con dos intervenciones manuales inevitables:

- carga del dataset entregado por la cátedra;
- carga temporal de credenciales de AstraDB.

El resto de las capas se regenera mediante código.

## 17.1 Quickstart de E2

Secuencia de ejecución:

- abrir `02_evaluacion_2_mvp.ipynb` en Google Colab;
- ejecutar las celdas en orden;
- cargar `cloud_provider_challenge_dataset_v1.zip` cuando se solicite;
- verificar los conteos de Batch → Bronze;
- permitir que Structured Streaming finalice con `availableNow`;
- revisar watermark, quality y quarantine;
- generar Silver y Gold;
- crear previamente en AstraDB el keyspace `cloud_provider_analytics`;
- cargar el Secure Connect Bundle y pegar el Application Token únicamente cuando Colab lo solicite;
- ejecutar la carga y las dos consultas CQL;
- revisar las pruebas de idempotencia;
- conservar las salidas del notebook como evidencia.

Para una ejecución completamente limpia puede eliminarse `/content/cloud_provider_e2` antes de comenzar. Para demostrar idempotencia no debe borrarse el checkpoint entre la primera ejecución y la prueba de re-ejecución.

## 17.2 Estimación en horas-persona

El feedback de E1 indicó que una distribución porcentual era menos accionable que una estimación real.

Para E2 se utiliza una estimación en horas-persona. Al tratarse de un proyecto individual, todas las actividades tienen como responsable a Nicolás Granata.

| Actividad | Responsable | Horas estimadas |
|---|---|---:|
| Incorporación de feedback y setup | Nicolás Granata | 2 |
| Batch → Bronze | Nicolás Granata | 3 |
| Structured Streaming y watermark | Nicolás Granata | 5 |
| Silver, calidad, quarantine y features | Nicolás Granata | 5 |
| Gold y pruebas de particionamiento | Nicolás Granata | 3 |
| AstraDB y CQL | Nicolás Granata | 4 |
| Idempotencia, pruebas y evidencias | Nicolás Granata | 3 |
| Documentación y cierre de E2 | Nicolás Granata | 3 |
| **Total estimado** | **Nicolás Granata** | **28** |

Las horas representan una estimación de planificación y no un registro automático de tiempo efectivo. Pueden ajustarse al finalizar la implementación.

## 17.3 Evidencia consolidada

Se genera un resumen JSON con los principales conteos de la ejecución.

Este archivo no reemplaza las salidas del notebook, pero facilita conservar una evidencia resumida dentro del repositorio.

In [95]:
import json

def variable_existe(nombre):
    return nombre in globals()


def contar_df(nombre):
    if not variable_existe(nombre):
        return None

    try:
        return globals()[nombre].count()
    except Exception as e:
        print(f"No se pudo contar {nombre}: {e}")
        return None


def comparar_variables(nombre_1, nombre_2):
    if not (
        variable_existe(nombre_1)
        and variable_existe(nombre_2)
    ):
        return None

    return bool(
        globals()[nombre_1]
        == globals()[nombre_2]
    )


# -----------------------------
# Fuentes
# -----------------------------

csv_count = (
    len(csv_files)
    if variable_existe("csv_files")
    else None
)

jsonl_count = (
    len(event_files)
    if variable_existe("event_files")
    else None
)


# -----------------------------
# Streaming Bronze
# -----------------------------

streaming_events_count = None

if (
    variable_existe("EVENTS_BRONZE_PATH")
    and os.path.exists(EVENTS_BRONZE_PATH)
):
    try:
        streaming_events_count = (
            spark.read
            .parquet(EVENTS_BRONZE_PATH)
            .count()
        )
    except Exception as e:
        print(
            "No se pudo leer Streaming Bronze:",
            e
        )


watermark_value = (
    int(WATERMARK_MINUTES)
    if variable_existe("WATERMARK_MINUTES")
    and WATERMARK_MINUTES is not None
    else None
)


# -----------------------------
# Idempotencia
# -----------------------------

batch_same_count = comparar_variables(
    "customers_count_before_rerun",
    "customers_count_after_rerun"
)

stream_same_count = comparar_variables(
    "stream_count_before_rerun",
    "stream_count_after_rerun"
)

controlled_dedupe_ok = comparar_variables(
    "dedupe_test_count",
    "raw_test_distinct"
)

astra_upsert_same_count = comparar_variables(
    "astra_count_before",
    "astra_count_after"
)


# -----------------------------
# Resumen consolidado
# -----------------------------

e2_summary = {
    "spark_version": str(spark.version),

    "sources": {
        "csv_files": csv_count,
        "jsonl_files": jsonl_count,
    },

    "batch_bronze": {
        "customers": contar_df(
            "customers_bronze_check"
        ),
        "users": contar_df(
            "users_bronze_check"
        ),
        "billing": contar_df(
            "billing_bronze_check"
        ),
    },

    "streaming_bronze": {
        "events": streaming_events_count,
        "watermark_minutes": watermark_value,
    },

    "quality": {
        "quarantine_rows": contar_df(
            "quarantine_check"
        ),
    },

    "silver": {
        "events": contar_df(
            "events_silver_check"
        ),
        "customers": contar_df(
            "customers_silver_check"
        ),
    },

    "gold": {
        "rows": contar_df(
            "gold_check"
        ),
        "duplicate_keys": (
            int(gold_duplicate_keys)
            if variable_existe(
                "gold_duplicate_keys"
            )
            and gold_duplicate_keys is not None
            else None
        ),
    },

    "idempotence": {
        "batch_same_count":
            batch_same_count,

        "stream_same_count":
            stream_same_count,

        "controlled_dedupe_ok":
            controlled_dedupe_ok,

        "astra_upsert_same_count":
            astra_upsert_same_count,
    },

    "astra": {
        "connection_completed":
            variable_existe("session"),

        "upsert_test_completed":
            astra_upsert_same_count
            is not None,
    },
}


# -----------------------------
# Guardado de evidencia
# -----------------------------

summary_path = (
    f"{EVIDENCE_PATH}/e2_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        e2_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        e2_summary,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "\nResumen guardado en:",
    summary_path
)


if variable_existe("cql_file"):
    print(
        "Script CQL:",
        cql_file
    )
else:
    print(
        "Script CQL: pendiente de completar "
        "la sección AstraDB."
    )

{
  "spark_version": "4.0.4",
  "sources": {
    "csv_files": 7,
    "jsonl_files": 120
  },
  "batch_bronze": {
    "customers": 80,
    "users": 800,
    "billing": 240
  },
  "streaming_bronze": {
    "events": 43200,
    "watermark_minutes": 86400
  },
  "quality": {
    "quarantine_rows": 2038
  },
  "silver": {
    "events": 41162,
    "customers": 80
  },
  "gold": {
    "rows": 11050,
    "duplicate_keys": 0
  },
  "idempotence": {
    "batch_same_count": true,
    "stream_same_count": true,
    "controlled_dedupe_ok": true,
    "astra_upsert_same_count": true
  },
  "astra": {
    "connection_completed": true,
    "upsert_test_completed": true
  }
}

Resumen guardado en: /content/cloud_provider_e2/evidence/e2_summary.json
Script CQL: /content/cloud_provider_e2/cql/02_evaluacion_2.cql


## 17.4 Tamaño físico de las capas generadas

Como evidencia adicional de ejecución se contabilizan los archivos y el espacio ocupado por las principales capas físicas del pipeline.

La medición corresponde al entorno temporal de Colab utilizado durante esta ejecución.

In [96]:
import os

def folder_stats(path):
    total_bytes = 0
    total_files = 0

    for root, dirs, files_in_dir in os.walk(path):
        for name in files_in_dir:
            file_path = os.path.join(root, name)

            if os.path.isfile(file_path):
                total_files += 1
                total_bytes += os.path.getsize(file_path)

    return total_files, total_bytes


paths_to_check = {
    "Bronze": BRONZE_PATH,
    "Silver": SILVER_PATH,
    "Gold": GOLD_PATH,
    "Quarantine": QUARANTINE_PATH,
}

total_all = 0

for label, path in paths_to_check.items():
    files_count, bytes_count = folder_stats(path)
    total_all += bytes_count

    print(
        f"{label}: "
        f"{files_count} archivos - "
        f"{bytes_count / (1024 * 1024):.2f} MB"
    )

print(
    "\nTotal:",
    f"{total_all / (1024 * 1024):.2f} MB"
)

Bronze: 11554 archivos - 36.42 MB
Silver: 726 archivos - 8.16 MB
Gold: 6 archivos - 0.20 MB
Quarantine: 362 archivos - 1.53 MB

Total: 46.31 MB


## 17.5 Descarga de artefactos de E2

Esta celda es manual y se utiliza al finalizar la ejecución para descargar los artefactos que serán versionados en GitHub.

No contiene datos sensibles ni credenciales.

In [97]:
from google.colab import files

print("Descargando script CQL...")
files.download(cql_file)

print("Descargando resumen de evidencia...")
files.download(summary_path)

Descargando script CQL...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando resumen de evidencia...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 18 - Backlog final y checklist de E2

La segunda evaluación debe cerrar con un backlog priorizado para la instancia final.

Las extensiones opcionales no deben poner en riesgo el funcionamiento end-to-end ni la reproducibilidad del MVP.

## 18.1 Backlog priorizado para la evaluación final

### Obligatorio

- incorporar cualquier corrección recibida en la devolución de E2;
- completar los marts Gold restantes requeridos para la instancia final;
- implementar las tablas Cassandra query-first necesarias para las cinco consultas finales;
- ejecutar y evidenciar las cinco consultas obligatorias;
- completar gobierno, observabilidad, manejo de errores y mecanismos de limpieza/reinicio;
- consolidar configuración externalizada y pruebas;
- actualizar arquitectura final y diccionario de datos;
- preparar presentación ejecutiva, demostración y video.

### Deseable

- ampliar pruebas automatizadas de calidad;
- comparar el percentil de anomalías con otro método simple;
- mejorar la generación automática de evidencias;
- optimizar tamaños de archivos cuando el volumen lo justifique.

### Fuera de alcance salvo necesidad posterior

- desplegar Kafka real;
- incorporar orquestadores externos;
- crear una infraestructura distribuida productiva;
- agregar tecnologías no requeridas por el caso o la cursada;
- realizar optimizaciones de gran escala que no puedan medirse con el dataset disponible.

## 18.2 Checklist de la segunda evaluación

| Requisito | Evidencia en este notebook |
|---|---|
| Correcciones de E1 incorporadas | Punto 1 |
| Arquitectura actualizada | Punto 12 |
| Tres fuentes batch en Bronze | Punto 3 |
| Streaming con esquema explícito | Punto 5 |
| Watermark decidido con evidencia | Punto 4 |
| Deduplicación por `event_id` | Puntos 5 y 15 |
| Late data | Punto 4 + watermark del Punto 5 |
| Checkpoint | Puntos 5 y 15 |
| Silver de eventos y maestro | Puntos 6 y 8 |
| Al menos tres features | Punto 8 |
| Tres reglas de calidad | Punto 7 |
| Quarantine | Punto 7 |
| `org_daily_usage_by_service` | Punto 10 |
| Particionamiento medido | Punto 11 |
| Componente analítico | Punto 9 |
| Cassandra/AstraDB poblada | Punto 13 |
| Dos consultas CQL | Punto 14 |
| Idempotencia demostrada | Punto 15 |
| Gobierno preliminar | Punto 16 |
| Quickstart y evidencia | Punto 17 |
| Esfuerzo en horas-persona | Punto 17 |
| Backlog final priorizado | Punto 18 |

## 18.3 Resultados consolidados de la implementación

La ejecución completa del MVP produjo los siguientes resultados:

| Evidencia | Resultado |
|---|---:|
| Fuentes CSV detectadas | 7 |
| Archivos JSONL procesados | 120 |
| Customers Bronze | 80 |
| Users Bronze | 800 |
| Billing Bronze | 240 |
| Eventos procesados por Structured Streaming | 43.200 |
| `event_id` distintos en Bronze | 43.200 |
| Eventos descartados por watermark | 0 |
| Watermark seleccionado | 86.400 min (60 días) |
| Registros enviados a quarantine | 2.038 |
| Eventos promovidos a Silver | 41.162 |
| Eventos Silver sin organización maestra | 0 |
| Filas del mart Gold | 11.050 |
| Claves duplicadas en Gold | 0 |
| Filas cargadas mediante upsert en AstraDB | 11.050 |
| Resultado de la consulta CQL de rango | 60 filas |
| Resultado de la consulta CQL de últimos días | 10 filas |
| Re-ejecución batch sin cambio de conteo | Sí |
| Re-ejecución streaming sin duplicación | Sí |
| Deduplicación controlada | Correcta |
| Upsert AstraDB idempotente | Sí |

Los **11.050 registros de Gold** coinciden además con los 11.050 grupos obtenidos en E1 mediante el flujo batch de referencia basado en `org_id + usage_date + service`.

Esta coincidencia permite comprobar que la implementación de E2 conserva el grano y la lógica definidos durante la etapa de diseño.

# Conclusión de la segunda evaluación

Esta segunda instancia transforma la fundación conceptual de E1 en un pipeline mínimo ejecutable.

La implementación incorpora procesamiento batch y streaming, Data Lake en Parquet, reglas de calidad, quarantine, features, un mart Gold FinOps, serving en Cassandra/AstraDB e idempotencia verificable.

Las decisiones que en E1 habían quedado abiertas se convierten en decisiones respaldadas por evidencia cuando el dataset lo permite. Cuando la información disponible no permite justificar una decisión productiva, la limitación se conserva explícita en lugar de reemplazarla por un valor arbitrario.

El resultado constituye la base técnica para ampliar los dominios analíticos y completar el MVP final.